# Inicializando

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType, DateType, IntegerType
from pyspark.sql.functions import col, trim, to_date

# Lendo dados da Tabela Bronze


In [0]:
df = spark.table('workspace.bronze.erp_cust_az12')
df.display()

# Remoção de espaços desnecessários em colunas string

In [0]:
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, trim(col(field.name)))
df.display()

# Limpeza para adequar a coluna CID aos dados da tabela crm_customers

In [0]:

df = df.withColumn(
    "cid",
    F.when(col("cid").startswith("NAS"),
           F.substring(col("cid"), 4, F.length(col("cid"))))
     .otherwise(col("cid"))
)

df.display()

# Verificação de validade das datas de nascimento

In [0]:
df = df.withColumn(
    'BDATE',
    F.when(col('BDATE') > F.current_date(), None)
    .otherwise(col('BDATE'))    
)

df.display()

# Normalização de valores da coluna GEN

In [0]:
df = ( 
    df.withColumn(
        'GEN', 
        F.when(F.upper(col("GEN")).isin('F', 'FEMALE'), 'Female')
        .when(F.upper(col("GEN")).isin('M', 'MALE'), 'Male')
        .otherwise('n/a')
    )
)

df.display()

# Renomeando colunas 

In [0]:
RENAME_MAP = {
    'CID': 'customer_id',
    'BDATE': 'birth_date',
    'GEN': 'gender'
}

for old_name, new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name, new_name)

df.limit(10).display()

# Criação de Tabela Silver

In [0]:
df.write.mode('overwrite').format('delta').saveAsTable('workspace.silver.erp_customers')

# Verificação de Dados na Tabela Final

In [0]:
%sql
SELECT * FROM workspace.silver.erp_customers LIMIT 10